<a href="https://colab.research.google.com/github/iamtrask/abcGPT/blob/main/notebooks/train_dual_alt_mixed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/></a>

# abcGPT dual-source — alt_mixed (two optimizers + pass-level alternation + mixed batches)

Each iter has a mixed batch (half-shake + half-wiki). Two separate AdamW optimizers, one per slot. Per pass (73 iters), only ONE optimizer steps; the other is fully frozen (state untouched). This combines mixed-batch's "each slot sees both corpora in forward" with alternating-mode's "clean per-slot Adam-state training windows."

If alternating's advantage was both per-pass corpus selection AND per-slot Adam isolation, this mode keeps only the second. The discriminating prediction: midpoint val drops near alternating's 1.85 and the slider produces readable text at both extremes.


## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Install dependencies

`torch` / `numpy` ship with Colab; `zstandard` does not.

In [ ]:
!pip install --quiet zstandard tiktoken

## 3. Clone abcGPT (shallow)

In [ ]:
import os
if os.path.isdir('/content/abcGPT'):
    # always pull latest main so a previously-cached clone doesn't ship stale code
    !cd /content/abcGPT && git fetch --depth 1 origin main && git reset --hard origin/main
else:
    !git clone --depth 1 --shallow-submodules https://github.com/iamtrask/abcGPT.git /content/abcGPT
%cd /content/abcGPT


## 4. Prepare the shakespeare_wiki_char dataset

Downloads tinyshakespeare and the first ~1.5 MB of wiki-abc, normalizes shakespeare to wiki surface form, concatenates with a `\n\n===\n\n` separator, and writes `train.bin` / `val.bin` / `meta.pkl`. The trainer partitions `train.bin` at that separator to serve wiki-only and shake-only batches; within a pass, every batch comes from the same corpus.

In [ ]:
!python data/shakespeare_wiki_char/prepare.py

## 5. Decide the run directory on Drive

A fresh `RUN_ID` is picked from the current timestamp on first run. If a Colab session dies mid-training, **re-mount Drive, set `RUN_ID = '<existing-dir>'` in this cell, and re-run the training cell** — the trainer detects existing pass snapshots in `out_dir` and resumes from the last completed pass (rolling optimizer state is saved alongside each snapshot for clean continuation).


In [ ]:
!ls /content/drive/MyDrive/abcGPT/runs

In [ ]:
RUN_ID = None   # set explicitly to resume an existing alt_mixed run, e.g. "20260519-XXXXXX-altmixed"


In [ ]:
import time, os, glob
DRIVE_ROOT = '/content/drive/MyDrive/abcGPT/runs'
os.makedirs(DRIVE_ROOT, exist_ok=True)
if 'RUN_ID' not in dir() or RUN_ID is None:
    RUN_ID = time.strftime('%Y%m%d-%H%M%S') + '-altmixed'
elif not RUN_ID.endswith('-altmixed'):
    RUN_ID = RUN_ID + '-altmixed'
OUT_DIR_DUAL = f'{DRIVE_ROOT}/{RUN_ID}'
os.makedirs(OUT_DIR_DUAL, exist_ok=True)
print('altmixed run dir:', OUT_DIR_DUAL)
snaps = sorted(glob.glob(os.path.join(OUT_DIR_DUAL, '*.pt.zst')))
if snaps:
    print(f'  found {len(snaps)} existing snapshots; trainer will resume')
else:
    print('  fresh run (no existing snapshots)')
other_runs = sorted([d for d in os.listdir(DRIVE_ROOT)
                     if d != RUN_ID and d.endswith('-altmixed')
                     and os.path.isdir(os.path.join(DRIVE_ROOT, d))])
if other_runs:
    print('\nother altmixed runs available:')
    for d in other_runs:
        n = len(glob.glob(os.path.join(DRIVE_ROOT, d, '*.pt.zst')))
        print(f"  {d}   ({n} snapshots)")


## 6. Sanity check: multi-pass round-trip on CPU

Runs a tiny GPT through 3 passes x 2 iters = 6 iters, saves pass snapshots and per-iter diffs, and checks (a) each saved snapshot matches in-memory state and (b) per-iter diffs roll up correctly between snapshots. Should print `OVERALL: PASS`.

In [ ]:
!python train_diff_logging.py --verify_roundtrip=True

## 7. Train

This is the long-running cell. The trainer first does a 5-iter warmup that prints step time vs diff-capture overhead. If overhead exceeds 20%, the trainer will suggest raising `--save_every`. Then it trains for 7500 iters by default in passes of 250 iters each, alternating shake -> wiki -> shake -> ..., writes a `diff_NNNNNN.pt.zst` per iter, a `pass_PPPP_<corpus>.pt.zst` at the end of each pass, plus `iter_log.jsonl` and `pass_log.jsonl`.

Knobs:
  - `--iters_per_pass=250` (smaller = more frequent alternation, more snapshots)
  - `--first_pass_corpus=shake` (or `wiki`)
  - `--save_every=1` (set to 2 or 4 if I/O is the bottleneck)
  - `--quantize_diffs=True` (int8 diffs, ~4x smaller, lossy; pass snapshots stay fp32)

In [ ]:
!python train_diff_logging.py config/train_shakespeare_wiki_char.py \
    --out_dir=$OUT_DIR \
    --iters_per_pass=73 \
    --first_pass_corpus=shake \
    --save_diffs=False


## 8. Listing + per-pass table

Lists the pass snapshots and per-iter diffs that landed on Drive, parses `pass_log.jsonl`, and prints `pass_idx | corpus | val_loss | snapshot_size`.

In [ ]:
import os, glob, json
diffs = sorted(glob.glob(os.path.join(OUT_DIR, 'diff_*.pt.zst')))
snaps = sorted(glob.glob(os.path.join(OUT_DIR, 'pass_*.pt.zst')))
iter_log = os.path.join(OUT_DIR, 'iter_log.jsonl')
pass_log = os.path.join(OUT_DIR, 'pass_log.jsonl')
meta = os.path.join(OUT_DIR, 'run_meta.json')

def total(paths):
    return sum(os.path.getsize(p) for p in paths)

diff_bytes = total(diffs)
snap_bytes = total(snaps)
print(f'run dir:         {OUT_DIR}')
print(f'pass snapshots:  {len(snaps)} files, {snap_bytes/1e9:.2f} GB')
print(f'per-iter diffs:  {len(diffs)} files, {diff_bytes/1e9:.2f} GB')
print(f'iter_log.jsonl:  exists={os.path.exists(iter_log)}')
print(f'pass_log.jsonl:  exists={os.path.exists(pass_log)}')
print(f'run_meta.json:   exists={os.path.exists(meta)}')
print(f'TOTAL:           {(diff_bytes+snap_bytes)/1e9:.2f} GB')

# per-pass table
if os.path.exists(pass_log):
    print()
    print(f"{'pass':>4} {'corpus':>6} {'val_loss':>10} {'snap_MB':>10}")
    print('-' * 36)
    with open(pass_log) as f:
        for line in f:
            row = json.loads(line)
            print(f"{row['pass_idx']:>4d} {row['corpus']:>6s} "
                  f"{row['val_loss_at_pass_end']:>10.4f} "
                  f"{row['snapshot_size_bytes']/1e6:>10.2f}")

## 9. Optional: reconstruct any iter's `state_dict`

Pick a target iter; load the nearest preceding pass snapshot (or `pass_init.pt.zst` if you want pre-pass-0 state), then apply per-iter diffs forward.

In [ ]:
import io, glob, os, re, zstandard, torch, json

def _decompress(b):
    return zstandard.ZstdDecompressor().decompress(b)

def _load_pt_zst(path):
    with open(path, 'rb') as f:
        blob = _decompress(f.read())
    return torch.load(io.BytesIO(blob), map_location='cpu', weights_only=False)

def load_state_at_iter(out_dir, target_iter):
    """Reconstruct state_dict at target_iter by starting from the nearest
    preceding pass snapshot (or pass_init for iters before pass 0 ends)
    and applying per-iter diffs forward."""
    # pass_PPPP_<corpus>.pt.zst — parse pass index from filename
    snap_paths = sorted(glob.glob(os.path.join(out_dir, 'pass_*.pt.zst')))
    candidates = []
    with open(os.path.join(out_dir, 'run_meta.json')) as f:
        meta = json.load(f)
    iters_per_pass = meta['iters_per_pass']
    for p in snap_paths:
        m = re.search(r'pass_(\d+)_', os.path.basename(p))
        if not m:
            continue
        pass_idx = int(m.group(1))
        end_iter = (pass_idx + 1) * iters_per_pass
        if end_iter <= target_iter:
            candidates.append((end_iter, p))
    if candidates:
        start_iter, snap_path = max(candidates)
    else:
        snap_path = os.path.join(out_dir, 'pass_init.pt.zst')
        start_iter = 0
    sd = _load_pt_zst(snap_path)['state_dict']
    for i in range(start_iter + 1, target_iter + 1):
        diff_path = os.path.join(out_dir, f'diff_{i:06d}.pt.zst')
        if not os.path.exists(diff_path):
            continue
        payload = _load_pt_zst(diff_path)
        diff = payload['diff']
        if payload.get('quantized'):
            diff = {k: v['q'].to(torch.float32) * v['scale'] for k, v in diff.items()}
        sd = {k: sd[k] + diff[k] for k in sd}
    return sd

# example: reconstruct iter 750
# sd = load_state_at_iter(OUT_DIR, 750)
# print({k: v.shape for k, v in list(sd.items())[:3]})

## 10. Weighted attribution sampling

Pick a base pass `K`. Above K, the model is treated as a shared base. After K, the per-pass diffs are split by which corpus that pass trained on. Scale the shake diffs by `α` and the wiki diffs by `β`, sum them onto `S_K`, and sample from the resulting model.

- `α = 1, β = 0` &rarr; "100% shakespeare" (only shake diffs after K are applied)
- `α = 0, β = 1` &rarr; "100% wikipedia"
- `α = β = 1` &rarr; reconstructs `S_N` exactly
- `α = β = 0` &rarr; just samples from `S_K`
- Negative coefficients are unlearning experiments (subtract a corpus's cumulative contribution)

When `K` is too small the early diffs are unstable and the weighted composition goes off-manifold (output noisy or collapsed). When `K` is too large the diffs are tiny and the slider barely moves the model. The interesting regime is in the middle.

In [ ]:
import io, glob, os, json, pickle, zstandard, torch
from ipywidgets import FloatSlider, IntSlider, Text, Button, Output, VBox
from IPython.display import display
from model import GPT, GPTConfig

# --- load per-pass snapshots + run metadata ---
RUN_META = json.load(open(os.path.join(OUT_DIR, 'run_meta.json')))
PASS_LOG = [json.loads(l) for l in open(os.path.join(OUT_DIR, 'pass_log.jsonl'))]

def _zload(path):
    with open(path, 'rb') as f:
        blob = zstandard.ZstdDecompressor().decompress(f.read())
    return torch.load(io.BytesIO(blob), map_location='cpu', weights_only=False)

snap_paths = [os.path.join(OUT_DIR, 'pass_init.pt.zst')] + sorted(glob.glob(os.path.join(OUT_DIR, 'pass_*.pt.zst')))
snap_paths = list(dict.fromkeys(snap_paths))   # de-dupe in case pass_init matched the glob
snaps   = [_zload(p)['state_dict'] for p in snap_paths]
# corpora[i] = corpus that produced the diff snaps[i-1] -> snaps[i]
corpora = [None] + [row['corpus'] for row in PASS_LOG]
print(f'{len(snaps)} snapshots loaded ({len(PASS_LOG)} passes; corpora: {sorted(set(c for c in corpora if c))})')

# --- precompute per-pass diffs ---
diffs = [None] + [{k: snaps[i][k] - snaps[i-1][k] for k in snaps[i]} for i in range(1, len(snaps))]

def weighted_state(K, alpha_shake, beta_wiki):
    """state = S_K + alpha * sum(shake-pass diffs after K) + beta * sum(wiki-pass diffs after K)"""
    state = {k: snaps[K][k].clone() for k in snaps[K]}
    for i in range(K + 1, len(snaps)):
        coef = alpha_shake if corpora[i] == 'shake' else beta_wiki
        if coef == 0.0:
            continue
        for k in diffs[i]:
            state[k] = state[k] + coef * diffs[i][k]
    return state

# --- model rebuild + char vocab from prepare.py output ---
META_PKL = pickle.load(open('data/shakespeare_wiki_char/meta.pkl', 'rb'))
stoi, itos = META_PKL['stoi'], META_PKL['itos']

gptconf = GPTConfig(**RUN_META['model_args'])
device  = 'cuda' if torch.cuda.is_available() else 'cpu'
model   = GPT(gptconf).eval().to(device)

def generate_from_state(state, prompt, max_new_tokens=300, temperature=0.8, top_k=40, seed=42):
    model.load_state_dict({k: v.to(device) for k, v in state.items()})
    torch.manual_seed(seed)
    ids = [stoi[c] for c in prompt if c in stoi] or [0]
    idx = torch.tensor([ids], dtype=torch.long, device=device)
    with torch.no_grad():
        out = model.generate(idx, max_new_tokens=max_new_tokens, temperature=temperature, top_k=top_k)
    return ''.join(itos.get(i.item(), '?') for i in out[0])

# --- interactive widget ---
N = len(snaps) - 1
K_slider     = IntSlider(value=N // 4, min=0, max=N, step=1, description='base K')
alpha_slider = FloatSlider(value=1.0, min=-1.0, max=2.0, step=0.05, description='alpha (shake)')
beta_slider  = FloatSlider(value=0.0, min=-1.0, max=2.0, step=0.05, description='beta (wiki)')
prompt_box   = Text(value='HAMLET:\n', description='prompt')
btn          = Button(description='Generate', button_style='primary')
out_widget   = Output(layout={'border': '1px solid #ddd', 'padding': '6px', 'width': '780px'})

def on_click(_=None):
    with out_widget:
        out_widget.clear_output()
        state = weighted_state(K_slider.value, alpha_slider.value, beta_slider.value)
        text  = generate_from_state(state, prompt_box.value)
        print(text)
        print(f'\n[K={K_slider.value} | alpha={alpha_slider.value:.2f} | beta={beta_slider.value:.2f} | seed=42]')

btn.on_click(on_click)
display(VBox([K_slider, alpha_slider, beta_slider, prompt_box, btn, out_widget]))
on_click()


## 11. Diff orthogonality across passes

If shake-passes and wiki-passes update approximately orthogonal subspaces of the model's weights, the (α, β) knobs above behave like independent dials and the per-source attribution is clean. If they share substantial overlap, turning α down quietly breaks β-controlled behavior. The cosine plot below makes this empirically visible across training.

- **Near 0**: diffs are disjoint, attribution is clean.
- **Near 1**: diffs are stepping on each other, attribution is mush.

Plotted is the cosine between each shake-pass diff and the wiki-pass diff that immediately followed (or preceded) it.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def _flat(d):
    return torch.cat([v.flatten() for v in d.values()])

def _cos(a, b):
    fa, fb = _flat(a), _flat(b)
    return (fa @ fb).item() / (fa.norm().item() * fb.norm().item() + 1e-12)

xs, ys = [], []
for i in range(1, len(diffs) - 1):
    if corpora[i] != corpora[i + 1]:
        xs.append(i)
        ys.append(_cos(diffs[i], diffs[i + 1]))

plt.figure(figsize=(11, 4))
plt.plot(xs, ys, marker='o', linewidth=1.2)
plt.axhline(0, color='gray', linewidth=0.6)
plt.xlabel('pass index')
plt.ylabel('cos(shake-diff, wiki-diff)')
plt.title('Adjacent shake / wiki diff cosine across training')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'mean cosine: {np.mean(ys):.4f}   median: {np.median(ys):.4f}   range: [{min(ys):.4f}, {max(ys):.4f}]')


## 12. Alternative: dual-source weight slots

The next cells run a parallel experiment that bakes the shake/wiki split into the **weights themselves**. Every parameter is held as two parallel tensors `W_s` and `W_w`, and the effective forward weight is `W = alpha * W_s + (1 - alpha) * W_w` with `alpha + beta = 1`. During training, `alpha` is resampled each iter from `Beta(0.5, 0.5)` so the optimizer has to make the interpolation work across the entire `alpha in [0, 1]` slider, not at a single mixing point. Gradient routing is per-source: shake batches only update `W_s`, wiki batches only update `W_w`. The mix knob is **structural** rather than recovered post-hoc from per-pass diffs (see `train_dual.py` and `model_dual.py`).

In [ ]:
# OUT_DIR_DUAL already points at this notebook's run dir (see section 5).
# This cell is kept for parity with train_diff_logging.ipynb's structure.
print('OUT_DIR_DUAL =', OUT_DIR_DUAL)


In [ ]:
!python train_dual.py config/train_shakespeare_wiki_dual_alt_mixed.py \
    --out_dir=$OUT_DIR_DUAL \
    --batch_mode=alt_mixed \
    --first_pass_corpus=shake


## 13. Single-slider inference widget

Under `alpha + beta = 1` there is a single knob, `alpha in [0, 1]`. The slider below moves smoothly from the wiki-only model (`alpha = 0`, `beta = 1`) to the shakespeare-only model (`alpha = 1`, `beta = 0`); `beta = 1 - alpha` is displayed alongside as a read-only label. The slider is strictly bounded to the trained range — no extrapolation.

In [ ]:
import io, glob, os, json, pickle, zstandard, torch
from ipywidgets import FloatSlider, Text, Button, Output, VBox, HBox, Label
from IPython.display import display
from model_dual import DualGPT, DualGPTConfig, set_mix

# --- load dual-run metadata and the latest pass snapshot ---
DUAL_META = json.load(open(os.path.join(OUT_DIR_DUAL, 'run_meta.json')))

def _zload(path):
    with open(path, 'rb') as f:
        blob = zstandard.ZstdDecompressor().decompress(f.read())
    return torch.load(io.BytesIO(blob), map_location='cpu', weights_only=False)

# Prefer final.pt.zst (run complete), fall back to latest.pt.zst (mid-run),
# fall back to legacy pass_*.pt.zst for backward compatibility.
_dual_load_path = None
for _candidate in ['final.pt.zst', 'latest.pt.zst']:
    _p = os.path.join(OUT_DIR_DUAL, _candidate)
    if os.path.exists(_p):
        _dual_load_path = _p
        break
if _dual_load_path is None:
    _legacy = sorted(glob.glob(os.path.join(OUT_DIR_DUAL, 'pass_[0-9]*_*.pt.zst')))
    if _legacy:
        _dual_load_path = _legacy[-1]
if _dual_load_path is None:
    raise RuntimeError(f'no dual checkpoint in {OUT_DIR_DUAL} (looked for final.pt.zst, latest.pt.zst, pass_*.pt.zst)')
print(f'loading dual checkpoint: {os.path.basename(_dual_load_path)}')
latest_sd = _zload(_dual_load_path)['state_dict']

# --- char vocab from prepare.py output ---
META_PKL = pickle.load(open('data/shakespeare_wiki_char/meta.pkl', 'rb'))
stoi, itos = META_PKL['stoi'], META_PKL['itos']

# --- rebuild DualGPT and load state ---
device = 'cuda' if torch.cuda.is_available() else 'cpu'
dual_cfg = DualGPTConfig(**DUAL_META['model_args'])
dual_model = DualGPT(dual_cfg).eval().to(device)
dual_model.load_state_dict({k: v.to(device) for k, v in latest_sd.items()})

def generate_dual(alpha, prompt, max_new_tokens=300, temperature=0.8, top_k=40, seed=42):
    set_mix(dual_model, float(alpha), 1.0 - float(alpha))
    torch.manual_seed(seed)
    ids = [stoi[c] for c in prompt if c in stoi] or [0]
    idx = torch.tensor([ids], dtype=torch.long, device=device)
    with torch.no_grad():
        out = dual_model.generate(idx, max_new_tokens=max_new_tokens,
                                  temperature=temperature, top_k=top_k)
    return ''.join(itos.get(i.item(), '?') for i in out[0])

# --- single-slider widget ---
alpha_slider = FloatSlider(value=0.5, min=0.0, max=1.0, step=0.02,
                           description='alpha', readout_format='.2f',
                           continuous_update=False)
beta_label   = Label(value='beta = 0.50   (wiki <- alpha=0 ... alpha=1 -> shake)')
prompt_box   = Text(value='KING:\n', description='prompt')
btn          = Button(description='Generate', button_style='primary')
out_widget   = Output(layout={'border': '1px solid #ddd', 'padding': '6px', 'width': '780px'})

def on_alpha(change):
    beta_label.value = (f'beta = {1.0 - alpha_slider.value:.2f}   '
                        f'(wiki <- alpha=0 ... alpha=1 -> shake)')
alpha_slider.observe(on_alpha, names='value')

def on_click(_=None):
    with out_widget:
        out_widget.clear_output()
        text = generate_dual(alpha_slider.value, prompt_box.value)
        print(text)
        print(f'\n[alpha={alpha_slider.value:.2f} | beta={1.0 - alpha_slider.value:.2f} | seed=42]')

btn.on_click(on_click)
display(VBox([HBox([alpha_slider, beta_label]), prompt_box, btn, out_widget]))
on_click()


### Cell A: per-corpus eval loss vs alpha

In [ ]:
# Cell A: per-corpus eval loss vs alpha curves
import numpy as np, torch, matplotlib.pyplot as plt

# Re-derive the shake/wiki split inside val.bin the same way prepare.py wrote it.
val_path = 'data/shakespeare_wiki_char/val.bin'
val_ids = np.memmap(val_path, dtype=np.uint16, mode='r')
SEPARATOR = '\n\n===\n\n'
sep_ids = np.array([stoi[c] for c in SEPARATOR], dtype=np.uint16)

def find_sep(arr, pat):
    n = len(pat)
    for i in range(len(arr) - n + 1):
        if np.array_equal(arr[i:i+n], pat):
            return i
    return -1

sep_idx = find_sep(np.asarray(val_ids), sep_ids)
print(f'val.bin: {len(val_ids):,} tokens; shake half [0:{sep_idx}], wiki half [{sep_idx + len(sep_ids)}:]')
shake_range = (0, sep_idx)
wiki_range  = (sep_idx + len(sep_ids), len(val_ids))

block_size = DUAL_META['model_args']['block_size']
batch_size = 8
eval_iters = 32

def batch_from(rng, seed):
    torch.manual_seed(seed)
    start, end = rng
    ix = torch.randint(end - start - block_size - 1, (batch_size,)) + start
    x = torch.stack([torch.from_numpy(val_ids[int(i):int(i)+block_size].astype(np.int64)) for i in ix]).to(device)
    y = torch.stack([torch.from_numpy(val_ids[int(i)+1:int(i)+1+block_size].astype(np.int64)) for i in ix]).to(device)
    return x, y

@torch.no_grad()
def eval_loss(alpha, rng):
    set_mix(dual_model, float(alpha), 1.0 - float(alpha))
    losses = []
    for k in range(eval_iters):
        x, y = batch_from(rng, seed=1337 + k)
        _, loss = dual_model(x, y)
        losses.append(float(loss.item()))
    return float(np.mean(losses))

alphas = np.linspace(0.0, 1.0, 21)
shake_curve = [eval_loss(a, shake_range) for a in alphas]
wiki_curve  = [eval_loss(a, wiki_range)  for a in alphas]

plt.figure(figsize=(9, 5))
plt.plot(alphas, shake_curve, marker='o', label='shake val loss')
plt.plot(alphas, wiki_curve,  marker='s', label='wiki val loss')
plt.xlabel('alpha  (0 = wiki only, 1 = shake only)')
plt.ylabel('cross-entropy loss')
plt.title('Dual-slot per-corpus eval loss vs alpha')
plt.grid(True, alpha=0.3); plt.legend()
plt.tight_layout(); plt.show()

print(f'{"alpha":>6s} | {"shake":>8s} | {"wiki":>8s}')
print('-' * 30)
for a, s, w in zip(alphas, shake_curve, wiki_curve):
    if round(a, 2) in (0.0, 0.25, 0.5, 0.75, 1.0):
        print(f'{a:>6.2f} | {s:>8.4f} | {w:>8.4f}')


### Cell B: comparison to alternating-pass baseline

In [ ]:
# Cell B: comparison to alternating-pass baseline
# Reuses `weighted_state(K, alpha_shake, beta_wiki)` from section 10. The
# baseline scheme can only mix per-pass diffs after a base pass K. For a like-
# for-like sweep we set K = N // 2 and let alpha_shake = alpha, beta_wiki = 1 - alpha,
# then evaluate the same shake/wiki val splits.
try:
    OUT_DIR
    have_baseline = ('snaps' in dir()) and ('diffs' in dir())
except NameError:
    have_baseline = False

if not have_baseline:
    print('Baseline alternating-pass run not loaded (section 10 cell not yet run). Skipping.')
else:
    from model import GPT, GPTConfig
    BASE_META = json.load(open(os.path.join(OUT_DIR, 'run_meta.json')))
    base_gptconf = GPTConfig(**BASE_META['model_args'])
    base_model = GPT(base_gptconf).eval().to(device)

    @torch.no_grad()
    def base_eval_loss(alpha, rng, K):
        state = weighted_state(K, alpha_shake=alpha, beta_wiki=1.0 - alpha)
        base_model.load_state_dict({k: v.to(device) for k, v in state.items()})
        losses = []
        for k in range(eval_iters):
            x, y = batch_from(rng, seed=1337 + k)
            _, loss = base_model(x, y)
            losses.append(float(loss.item()))
        return float(np.mean(losses))

    N = len(snaps) - 1
    K = N // 2
    base_shake = [base_eval_loss(a, shake_range, K) for a in alphas]
    base_wiki  = [base_eval_loss(a, wiki_range,  K) for a in alphas]

    plt.figure(figsize=(9, 5))
    plt.plot(alphas, shake_curve, marker='o', linestyle='-',  label='dual: shake val')
    plt.plot(alphas, wiki_curve,  marker='s', linestyle='-',  label='dual: wiki val')
    plt.plot(alphas, base_shake,  marker='o', linestyle='--', label=f'baseline (K={K}): shake val')
    plt.plot(alphas, base_wiki,   marker='s', linestyle='--', label=f'baseline (K={K}): wiki val')
    plt.xlabel('alpha  (0 = wiki only, 1 = shake only)')
    plt.ylabel('cross-entropy loss')
    plt.title('Dual-slot vs alternating-pass post-hoc mixing')
    plt.grid(True, alpha=0.3); plt.legend()
    plt.tight_layout(); plt.show()


### Cell C: W_s vs W_w cosine histogram

In [ ]:
# Cell C: weight-space cosine histogram between W_s and W_w
import numpy as np, matplotlib.pyplot as plt, torch
from model_dual import DualLinear, DualEmbedding, DualLayerNorm

cosines, names = [], []
with torch.no_grad():
    for name, m in dual_model.named_modules():
        if isinstance(m, (DualLinear, DualEmbedding, DualLayerNorm)):
            fs = m.W_s.detach().flatten().float().cpu()
            fw = m.W_w.detach().flatten().float().cpu()
            denom = fs.norm().item() * fw.norm().item() + 1e-12
            c = (fs @ fw).item() / denom
            cosines.append(c); names.append(name)

cosines = np.array(cosines)
plt.figure(figsize=(9, 4))
plt.hist(cosines, bins=20, edgecolor='black')
plt.axvline(1.0, color='red', linestyle='--', label='cos = 1.0 (identical, iter 0)')
plt.xlabel('cos(W_s, W_w)'); plt.ylabel('count')
plt.title('Per-parameter cosine between shake and wiki slots after training')
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

print(f'n params with both slots : {len(cosines)}')
print(f'mean cosine              : {cosines.mean():.4f}')
print(f'median cosine            : {float(np.median(cosines)):.4f}')
print(f'range                    : [{cosines.min():.4f}, {cosines.max():.4f}]')
print('Note: W_s and W_w were byte-identical at iter 0 (cosine = 1.0). Lower cosine means the')
print('two slots have moved further apart, i.e. the gradient routing carved more orthogonal subspaces.')


### Cell D: Pareto frontier scatter

In [ ]:
# Cell D: Pareto frontier across alpha
import numpy as np, matplotlib.pyplot as plt

plt.figure(figsize=(7, 7))
plt.scatter(shake_curve, wiki_curve, c=alphas, cmap='viridis', s=60, edgecolor='black')
for target in (0.0, 0.5, 1.0):
    idx = int(np.argmin(np.abs(alphas - target)))
    plt.annotate(f'alpha={target}', (shake_curve[idx], wiki_curve[idx]),
                 textcoords='offset points', xytext=(8, 6), fontsize=10)
cbar = plt.colorbar(); cbar.set_label('alpha')
plt.xlabel('shake val loss'); plt.ylabel('wiki val loss')
plt.title('Pareto frontier traced by alpha sweep')
plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()


### Cell E: qualitative 5-row generation table

In [ ]:
# Cell E: qualitative samples across alpha
import torch
rows = []
for alpha in (0.0, 0.25, 0.5, 0.75, 1.0):
    text = generate_dual(alpha, 'KING:\n', max_new_tokens=200, temperature=0.8, top_k=40, seed=42)
    head = text.replace('\n', ' ')[:120]
    rows.append((alpha, head))

print(f'{"alpha":>6s} | first 120 chars after \"KING:\\n\" prompt')
print('-' * 130)
for alpha, head in rows:
    print(f'{alpha:>6.2f} | {head}')
